# 🏥 RIVA — Triage Model Training (PIMA)
**الهدف:** تدريب XGBoost على PIMA Dataset → تصدير ONNX

| المقياس | النتيجة |
|---------|--------|
| Accuracy | 74.7% |
| F1-Score | 68.3% |
| ROC-AUC | 83.5% |
| CV F1 | 80.4% |
| CV AUC | 88.4% |

## 1️⃣ تثبيت

In [ ]:
!pip install xgboost scikit-learn imbalanced-learn onnxmltools onnxruntime -q
print('OK')

## 2️⃣ رفع الداتا

In [ ]:
from google.colab import files
uploaded = files.upload()
import pandas as pd
df = pd.read_csv('diabetes.csv')
print(f'OK: {df.shape}')

## 3️⃣ EDA

In [ ]:
print('=== توزيع النتيجة ===')
print(df['Outcome'].value_counts())
print(f'نسبة السكري: {round(df["Outcome"].mean()*100,1)}%')
zero_cols = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
print('\n=== القيم الصفرية ===')
for col in zero_cols:
    z = (df[col]==0).sum()
    print(f'  {col}: {z} ({round(z/len(df)*100,1)}%)')

## 4️⃣ تنظيف + KNNImputer

In [ ]:
import numpy as np
from sklearn.impute import KNNImputer

df[zero_cols] = df[zero_cols].replace(0, np.nan)
FEATURES = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
            'Insulin','BMI','DiabetesPedigreeFunction','Age']
imputer = KNNImputer(n_neighbors=5)
df[FEATURES] = imputer.fit_transform(df[FEATURES])
print(f'Missing after imputation: {df.isnull().sum().sum()}')

## 5️⃣ SMOTE + Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

X = df[FEATURES].values.astype(np.float32)
y = df['Outcome'].values
X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train_raw, y_train_raw)
print(f'After SMOTE: {np.bincount(y_res)}')
scaler = StandardScaler()
X_train  = scaler.fit_transform(X_res).astype(np.float32)
X_test_s = scaler.transform(X_test).astype(np.float32)
print(f'Train: {len(X_train)} | Test: {len(X_test_s)}')

## 6️⃣ تدريب XGBoost

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_auc_score

model = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9,
    scale_pos_weight=1, random_state=42,
    eval_metric='aucpr', n_jobs=-1
)
model.fit(X_train, y_res, eval_set=[(X_test_s, y_test)], verbose=False)
y_pred  = model.predict(X_test_s)
y_proba = model.predict_proba(X_test_s)[:,1]
acc = round(accuracy_score(y_test,y_pred)*100,1)
f1  = round(f1_score(y_test,y_pred)*100,1)
auc = round(roc_auc_score(y_test,y_proba)*100,1)
print(f'Accuracy: {acc}% | F1: {f1}% | ROC-AUC: {auc}%')
print(classification_report(y_test,y_pred,target_names=['No Diabetes','Diabetes']))

## 7️⃣ تصدير ONNX

In [ ]:
from onnxmltools import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType
import onnxruntime as rt, json, os, pickle

os.makedirs('ai-core/models/triage', exist_ok=True)
initial_type = [('float_input', FloatTensorType([None, len(FEATURES)]))]
onnx_model   = convert_xgboost(model, initial_types=initial_type)
with open('ai-core/models/triage/model_int8.onnx','wb') as f:
    f.write(onnx_model.SerializeToString())
with open('ai-core/models/triage/imputer.pkl','wb') as f:
    pickle.dump(imputer, f)
with open('ai-core/models/triage/scaler.pkl','wb') as f:
    pickle.dump(scaler, f)
feat_imp = sorted(zip(FEATURES,model.feature_importances_),key=lambda x:x[1],reverse=True)
features_data = {
    'features': FEATURES, 'target': 'Outcome',
    'n_features': len(FEATURES),
    'classes': ['No Diabetes','Diabetes'],
    'metrics': {'accuracy':acc,'f1_score':f1,'roc_auc':auc},
    'top_features': [f[0] for f in feat_imp[:5]],
    'imputation': 'KNNImputer (n_neighbors=5)',
    'scaler': {'mean':scaler.mean_.tolist(),'std':scaler.scale_.tolist()}
}
with open('ai-core/models/triage/features.json','w',encoding='utf-8') as f:
    json.dump(features_data,f,ensure_ascii=False,indent=2)
print(f'OK: {round(os.path.getsize("ai-core/models/triage/model_int8.onnx")/1024,1)} KB')

## 8️⃣ اختبار + تحميل

In [ ]:
sess = rt.InferenceSession('ai-core/models/triage/model_int8.onnx')
inp  = sess.get_inputs()[0].name
test_cases = [
    ([6,148,72,35,0,33.6,0.627,50],'Diabetes'),
    ([1,85,66,29,0,26.6,0.351,31],'No Diabetes'),
]
for feat,exp in test_cases:
    df_t = pd.DataFrame([feat],columns=FEATURES)
    x    = scaler.transform(imputer.transform(df_t)).astype(np.float32)
    pred = sess.run(None,{inp:x})[0][0]
    res  = 'Diabetes' if pred==1 else 'No Diabetes'
    print(f'  {"OK" if res==exp else "FAIL"}: {res}')

import shutil
from google.colab import files
for f in ['ai-core/models/triage/model_int8.onnx',
          'ai-core/models/triage/features.json',
          'ai-core/models/triage/imputer.pkl',
          'ai-core/models/triage/scaler.pkl']:
    fname = f.split('/')[-1]
    shutil.copy(f, f'/content/drive/MyDrive/RIVA_PROJECT_BACKUP/{fname}')
    files.download(f)
print('OK: all files saved')